In [34]:
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader

In [35]:
class space_data(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features.to_numpy(), dtype=torch.float32)
        self.labels = torch.tensor(labels.to_numpy(), dtype = torch.float32)
    
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [36]:
df = pd.read_csv("data/cleaned_combined.csv")
target = "Transported"
X = df.drop(columns= target)
y = df[target]

In [37]:
X.shape, y.shape

((12970, 22), (12970,))

In [38]:
X_train,X_test = X[:-4277], X[-4277:]
y_train,y_test = y[:-4277], y[-4277:]

In [39]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8693, 22), (4277, 22), (8693,), (4277,))

In [40]:
X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CryoSleep                  8693 non-null   int64  
 1   Age                        8693 non-null   float64
 2   VIP                        8693 non-null   int64  
 3   Side                       8693 non-null   int64  
 4   Group                      8693 non-null   int64  
 5   Spendings                  8693 non-null   float64
 6   HomePlanet_Earth           8693 non-null   int64  
 7   HomePlanet_Europa          8693 non-null   int64  
 8   HomePlanet_Mars            8693 non-null   int64  
 9   Destination_55 Cancri e    8693 non-null   int64  
 10  Destination_PSO J318.5-22  8693 non-null   int64  
 11  Destination_TRAPPIST-1e    8693 non-null   int64  
 12  Deck_A                     8693 non-null   int64  
 13  Deck_B                     8693 non-null   int64  
 14  Dec

In [41]:
from sklearn.preprocessing import StandardScaler

cols_to_be_scaled = ["Age", "Spendings"]

ss = StandardScaler()
X_train[cols_to_be_scaled] = ss.fit_transform(X_train[cols_to_be_scaled])
X_test[cols_to_be_scaled] = ss.transform(X_test[cols_to_be_scaled])

In [42]:
train_dataset = space_data(X_train, y_train)
test_dataset = space_data(X_test, y_test)

In [43]:
train_dataset[0]

(tensor([ 0.0000,  0.7036,  0.0000,  0.0000,  1.0000, -0.5190,  0.0000,  1.0000,
          0.0000,  0.0000,  0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  1.0000,  1.0000]),
 tensor(0.))

In [44]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [45]:
class NN(nn.Module):
    def __init__(self, num_features):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

In [46]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [52]:
lr = 0.001
epochs = 200

In [53]:
model = NN(X_train.shape[1])
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

In [54]:
train_losses = []

In [55]:
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for x, y in train_dataloader:
        x, y = x.to(device), y.float().unsqueeze(1).to(device)

        outputs = model(x)
    
        loss = criterion(outputs, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_dataloader)
    train_losses.append(avg_loss)
    print(f"Epoch: {epoch+1} | Loss: {avg_loss}")

Epoch: 1 | Loss: 0.6969810009879225
Epoch: 2 | Loss: 0.688626757439445
Epoch: 3 | Loss: 0.6840506976141649
Epoch: 4 | Loss: 0.6779159704113707
Epoch: 5 | Loss: 0.6649307722554487
Epoch: 6 | Loss: 0.6581205738817945
Epoch: 7 | Loss: 0.6400841484394144
Epoch: 8 | Loss: 0.6297442115405026
Epoch: 9 | Loss: 0.6192602931576616
Epoch: 10 | Loss: 0.610214282922885
Epoch: 11 | Loss: 0.6016060711048982
Epoch: 12 | Loss: 0.6113465249757556
Epoch: 13 | Loss: 0.6046177765683216
Epoch: 14 | Loss: 0.5963719971477985
Epoch: 15 | Loss: 0.5895162298179725
Epoch: 16 | Loss: 0.584393135874587
Epoch: 17 | Loss: 0.5920269771972123
Epoch: 18 | Loss: 0.5821647790863234
Epoch: 19 | Loss: 0.5723251337733339
Epoch: 20 | Loss: 0.5800969034214231
Epoch: 21 | Loss: 0.5845814411911894
Epoch: 22 | Loss: 0.5770323477028048
Epoch: 23 | Loss: 0.5777285972281414
Epoch: 24 | Loss: 0.575674830661977
Epoch: 25 | Loss: 0.5820491570321953
Epoch: 26 | Loss: 0.5683891015017734
Epoch: 27 | Loss: 0.564540663624511
Epoch: 28 | Los

In [56]:
model.eval()
total_loss = correct = total = 0
with torch.inference_mode():
    
    for x, y in test_dataloader:
        x, y = x.to(device), y.float().unsqueeze(1).to(device)

        outputs = model(x)
        loss = criterion(outputs, y)

        total_loss += loss.item()

        probs = torch.sigmoid(outputs)

        preds = (probs > 0.5).float()

        correct += (preds == y).sum().item()

        total += y.size(0)

avg_loss = total_loss / len(test_dataloader)
accuracy = correct / total 

print(f"val loss: {avg_loss} | accuracy: {accuracy}")

            

val loss: 0.9808226321170579 | accuracy: 0.6509235445405658


In [57]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

logistic = LogisticRegression(
    max_iter=1000
)

logistic.fit(X_train, y_train)

preds = logistic.predict(X_test)

acc = accuracy_score(preds, y_test)

acc

0.5784428337619827